# Phase 3 — Data Acquisition and Validation

## Goal

Inspect the Kaggle crop-yield dataset, identify risks, and create one validated observation per country, crop, and year before exploratory analysis or modelling.


## Setup

Run this notebook from the repository root after placing the five source CSV files in `data/raw/`. Raw and generated datasets are ignored by Git.


In [ ]:
from pathlib import Path

import pandas as pd

from crop_yield.data import (
    load_yield_data,
    prepare_modeling_data,
    save_processed_data,
)

RAW_PATH = Path("data/raw/yield_df.csv")
PROCESSED_PATH = Path("data/processed/crop_yield_modeling.csv")


## Steps

### 1. Load and preview the source data


In [ ]:
raw_data = load_yield_data(RAW_PATH)
raw_data.head()


### 2. Confirm shape, columns and data types


In [ ]:
source_profile = pd.DataFrame({
    "dtype": raw_data.dtypes.astype(str),
    "missing_count": raw_data.isna().sum(),
    "unique_count": raw_data.nunique(dropna=False),
})
print(f"Rows: {len(raw_data):,}")
print(f"Columns: {raw_data.shape[1]}")
source_profile


### 3. Test the intended grain

The intended grain is one country (`Area`), crop (`Item`) and year.


In [ ]:
grain = ["Area", "Item", "Year"]
duplicate_grain_mask = raw_data.duplicated(grain, keep=False)
duplicate_group_count = raw_data.groupby(grain).size().gt(1).sum()

pd.Series({
    "rows_in_repeated_groups": int(duplicate_grain_mask.sum()),
    "repeated_country_crop_year_groups": int(duplicate_group_count),
    "share_of_rows_in_repeated_groups": float(duplicate_grain_mask.mean()),
})


### 4. Identify what changes inside repeated groups

Yield, rainfall and pesticide values should not conflict within the same country-crop-year. Temperature can have multiple observations and will be averaged after exact semantic duplicates are removed.


In [ ]:
repeated = raw_data.loc[duplicate_grain_mask].drop(
    columns=["Unnamed: 0"], errors="ignore"
)
fields_to_check = [
    "hg/ha_yield",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes",
    "avg_temp",
]
variation_by_field = {
    field: int(repeated.groupby(grain)[field].nunique().gt(1).sum())
    for field in fields_to_check
}
pd.Series(variation_by_field, name="groups_with_multiple_values")


### 5. Create and save the validated modelling table


In [ ]:
modeling_data = prepare_modeling_data(raw_data)
saved_path = save_processed_data(modeling_data, PROCESSED_PATH)

print(f"Validated rows: {len(modeling_data):,}")
print(f"Saved locally to: {saved_path}")
modeling_data.head()


## Checks


In [ ]:
assert len(raw_data) == 28_242
assert len(modeling_data) == 13_130
assert modeling_data.shape[1] == 7
assert not modeling_data.duplicated(["area", "item", "year"]).any()
assert not modeling_data.isna().any().any()
assert 2003 not in modeling_data["year"].unique()
print("All Phase 3 validation checks passed.")


## Next Steps

- Explore distributions and relationships without making causal claims.
- Choose a leakage-resistant validation strategy.
- Establish a simple regression baseline before complex models.

**Known limitation:** 2003 is absent, and the merged data may exclude source observations that failed to match across files.
